# ET partitioning: irrigation vs shallow-GW/soil moisture

Final step: estimate irrigation-attributable ET (ET_irr) using donor-based ET_gwsm within terrain/water-source strata.

## Connection to prior notebooks

This notebook consumes outputs from NB02, NB04, and NB05:

| Input | Source | Description |
|---|---|---|
| `fields_pattern.fgb` | NB04 | Field polygons with `strata` and `pattern` columns |
| `et_join/{FID}.parquet` | NB05 | Daily GridMET + interpolated PT-JPL ETf per field |

Only fields with `strata != "non_partitioned"` are partitioned. Pattern donors (identified in NB04 via IrrMapper) provide the `etf_gwsm` background signal transferred to recipient fields within the same stratum.

In [ ]:
# Shared color palette — used throughout all notebooks
STREAM_BLUE = "#2171b5"
WET_CMAP = "Blues"
TERRAIN_CMAP = "terrain"
REM_CMAP = "RdYlBu_r"
NDWI_CMAP = "RdYlGn"
STRATA_COLORS = {
    "perennial": "#2166ac",
    "intermittent": "#74add1",
    "managed": "#4dac26",
    "non_partitioned": "#d9d9d9",
}
PARTITION_COLORS = {"pe": "#a6cee3", "et_gwsm": "#1f78b4", "et_irr": "#e31a1c"}

## 1. Objective

Where shallow groundwater can subsidize ET, satellite ETa alone does not map directly to irrigation consumptive use. We partition ETa into `PE`, `ET_gwsm`, and `ET_irr` using stratum-matched non-irrigated donors.

## 2. Setup

In [ ]:
import os
import tomllib
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from handily.config import HandilyConfig

# Load configuration
config_path = Path("beaverhead_config.toml")
with open(config_path, "rb") as f:
    cfg = tomllib.load(f)

config = HandilyConfig.from_dict(cfg)
out_dir = os.path.expanduser(config.out_dir)

print(f"Joined parquet dir: {config.partition_joined_parquet_dir}")
print(f"Partitioned output dir: {config.partition_out_parquet_dir}")

## 3. Water-balance formulation

`ETa = PE + ET_gwsm + ET_irr`

- `ETa`: daily/annual actual ET from PT-JPL × GridMET ETo
- `PE`: effective precipitation (here `min(P, ETa)`)
- `ET_gwsm`: background ET supported by shallow GW/soil moisture
- `ET_irr`: irrigation-attributable ET (residual)

`ET_gwsm` is estimated from pattern fields (non-irrigated) within the same stratum; `ET_irr` is then computed as the residual.

In [ ]:
# Donor-recipient field map
pattern_path = os.path.join(out_dir, "fields_pattern.fgb")
fields_pattern = None
if os.path.exists(pattern_path):
    fields_pattern = gpd.read_file(pattern_path)

if fields_pattern is not None and "pattern" in fields_pattern.columns and "strata" in fields_pattern.columns:
    donors = fields_pattern[fields_pattern["pattern"] == True]
    recipients = fields_pattern[
        (fields_pattern["pattern"] == False) &
        (fields_pattern["strata"] != "non_partitioned")
    ]
    non_part = fields_pattern[fields_pattern["strata"] == "non_partitioned"]

    fig, ax = plt.subplots(figsize=(10, 8))
    non_part.plot(ax=ax, color="#d9d9d9", edgecolor="black", linewidth=0.2, label=f"Non-partitioned (n={len(non_part)})")
    recipients.plot(ax=ax, color=STREAM_BLUE, alpha=0.5, edgecolor="black", linewidth=0.2, label=f"Recipients (n={len(recipients)})")
    donors.plot(ax=ax, color="green", alpha=0.7, edgecolor="black", linewidth=0.2, label=f"Donors (n={len(donors)})")
    ax.set_title("Donor-Recipient Field Classification")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] Pattern fields not yet computed — run NB04 first")

In [ ]:
# Donor count by strata
if fields_pattern is not None and "pattern" in fields_pattern.columns and "strata" in fields_pattern.columns:
    donor_counts = fields_pattern[fields_pattern["pattern"] == True].groupby("strata").size()
    if len(donor_counts) > 0:
        fig, ax = plt.subplots(figsize=(6, 4))
        donor_counts.plot.bar(
            ax=ax,
            color=[STRATA_COLORS.get(s, "#aaa") for s in donor_counts.index],
            edgecolor="black",
        )
        ax.set_title("Pattern Donors by Strata")
        ax.set_ylabel("Count")
        ax.set_xlabel("Strata")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
    else:
        print("[SKIP] No donor fields found in pattern data")
else:
    print("[SKIP] Pattern fields not available")

## 4. Donor (pattern) fields

For each irrigated field, select the nearest pattern field in the same stratum and transfer an ETo-normalized background signal:

```
etf_gwsm = (ETa_donor - PE_donor) / ETo_donor
ET_gwsm  = etf_gwsm × ETo_recipient
ET_irr   = ETa - PE - ET_gwsm
```

Same-stratum matching constrains donors to comparable groundwater-access setting (REM) and water-source type.

In [ ]:
# Load fields with pattern column
pattern_path = os.path.join(out_dir, "fields_pattern.fgb")

if os.path.exists(pattern_path):
    fields = gpd.read_file(pattern_path)
    
    if 'strata' in fields.columns and 'pattern' in fields.columns:
        # Show pattern distribution by strata
        print("Pattern fields per strata:")
        print(fields.groupby('strata')['pattern'].agg(['sum', 'count']))
        print(f"\nTotal pattern fields: {fields['pattern'].sum()}")
        print(f"Total irrigated fields: {(~fields['pattern']).sum()}")
else:
    print(f"Pattern fields not found at {pattern_path}")
    print("Run the pattern selection step.")

In [ ]:
from handily.et.partition import partition_et

if config.run_partition:
    results = partition_et(
        fields_path=config.fields_path,
        joined_parquet_dir=config.partition_joined_parquet_dir,
        out_parquet_dir=config.partition_out_parquet_dir,
        feature_id=config.feature_id,
        strata_col=config.partition_strata_col,
        pattern_col=config.partition_pattern_col,
    )
    print(f"Partitioned {results['n_fields']} fields, {results['n_donors']} donors")
else:
    print("[SKIP] run_partition=False — loading cached partition parquet below")
    part_dir = os.path.expanduser(config.partition_out_parquet_dir) if config.partition_out_parquet_dir else None
    if part_dir and os.path.exists(part_dir):
        files = [f for f in os.listdir(part_dir) if f.endswith('.parquet')]
        print(f"  Found {len(files)} cached partitioned parquet files")

In [ ]:
# Demonstrate the partitioning calculation

# Simulated data for an irrigated field and its donor
np.random.seed(42)

# Annual values for a water year
eto_annual = 1200  # mm/year reference ET
prcp_annual = 300  # mm/year precipitation

# Donor (pattern field) - no irrigation
aet_donor = 400  # mm/year (natural ET)
pe_donor = min(prcp_annual, aet_donor)  # 300 mm
et_gwsm_donor = aet_donor - pe_donor  # 100 mm from groundwater
etf_gwsm = et_gwsm_donor / eto_annual  # fraction

print("=== Donor (Pattern) Field ===")
print(f"AET: {aet_donor} mm/year")
print(f"PE: {pe_donor} mm/year")
print(f"ET_gwsm: {et_gwsm_donor} mm/year")
print(f"ETf_gwsm: {etf_gwsm:.3f}")

# Recipient (irrigated field)
aet_recipient = 800  # mm/year (higher due to irrigation)
pe_recipient = min(prcp_annual, aet_recipient)  # 300 mm
et_gwsm_recipient = etf_gwsm * eto_annual  # apply donor fraction
et_irr = aet_recipient - et_gwsm_recipient - pe_recipient

print("\n=== Recipient (Irrigated) Field ===")
print(f"AET: {aet_recipient} mm/year")
print(f"PE: {pe_recipient} mm/year")
print(f"ET_gwsm (from donor): {et_gwsm_recipient:.0f} mm/year")
print(f"ET_irr (calculated): {et_irr:.0f} mm/year")

# Verification
total = pe_recipient + et_gwsm_recipient + et_irr
print(f"\nVerification: PE + ET_gwsm + ET_irr = {total:.0f} mm/year (should equal AET={aet_recipient})")

## 6. Running the Partition

In [ ]:
# Check if partitioned data exists
part_dir = os.path.expanduser(config.partition_out_parquet_dir) if config.partition_out_parquet_dir else None

if part_dir and os.path.exists(part_dir):
    parquet_files = [f for f in os.listdir(part_dir) if f.endswith('.parquet')]
    print(f"Found {len(parquet_files)} partitioned parquet files")
    
    if parquet_files:
        example_file = os.path.join(part_dir, parquet_files[0])
        part_df = pd.read_parquet(example_file)
        
        print(f"\nExample file: {parquet_files[0]}")
        print(f"Columns: {list(part_df.columns)}")
        print(f"Records: {len(part_df)}")
        
        part_df.head()
else:
    print("Partitioned parquet directory not found.")
    print("Run the ET partition step.")

In [ ]:
# Example: Running the partition
# (Commented out to avoid re-running if already complete)

# from handily.et.partition import partition_et
#
# result = partition_et(
#     fields_path=os.path.join(out_dir, "fields_pattern.fgb"),
#     joined_parquet_dir=config.partition_joined_parquet_dir,
#     out_parquet_dir=config.partition_out_parquet_dir,
#     feature_id=config.feature_id,
#     strata_col=config.partition_strata_col,
#     pattern_col=config.partition_pattern_col,
# )
#
# print(f"Partitioned {result['n_fields']} fields")
# print(f"Using {result['n_donors']} donor fields")
# print(f"Strata distribution: {result['strata_counts']}")

print("See beaverhead.py for the full workflow script.")

In [ ]:
# Annual stacked bar: PE + ET_gwsm + ET_irr
part_dir = os.path.expanduser(config.partition_out_parquet_dir) if config.partition_out_parquet_dir else None
part_df = None
if part_dir and os.path.exists(part_dir):
    pfiles = [f for f in os.listdir(part_dir) if f.endswith('.parquet')]
    if pfiles:
        part_df = pd.read_parquet(os.path.join(part_dir, pfiles[0]))

if part_df is not None and "water_year" in part_df.columns:
    has_pe = "pe" in part_df.columns
    has_gwsm = "et_gwsm_m" in part_df.columns
    has_irr = "et_irr_m" in part_df.columns

    if has_gwsm and has_irr:
        cols = []
        colors = []
        labels = []
        if has_pe:
            cols.append("pe")
            colors.append(PARTITION_COLORS["pe"])
            labels.append("PE")
        cols.append("et_gwsm_m")
        colors.append(PARTITION_COLORS["et_gwsm"])
        labels.append("ET_gwsm")
        cols.append("et_irr_m")
        colors.append(PARTITION_COLORS["et_irr"])
        labels.append("ET_irr")

        annual = part_df.groupby("water_year")[cols].sum()
        fig, ax = plt.subplots(figsize=(10, 5))
        annual[cols].plot.bar(ax=ax, stacked=True, color=colors)
        ax.set_title("Annual ET Components (mean across fields)")
        ax.set_ylabel("mm/year")
        ax.legend(labels)
        ax.set_xlabel("Water Year")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("Expected columns (et_gwsm_m, et_irr_m) not found in partition data")
else:
    print("[SKIP] Partition data not yet computed — run partition_et first")

### Note on negative irrigation clipping

When the donor `etf_gwsm` transfer produces `et_irr < 0` (the donor field has higher background ET than the recipient), the negative value is clipped to 0 and `et_gwsm` absorbs the residual. This keeps the water balance closed:

```
ET_irr_raw = ETa - PE - ET_gwsm
if ET_irr_raw < 0:
    ET_gwsm = ETa - PE  # absorb residual into gwsm
    ET_irr = 0
```

This situation is most common for small-REM recipient fields near perennial streams where the donor may have unusually high background ET in a particular year.

In [ ]:
# Monthly ET component time series (representative single-field view)
if part_df is not None and "water_year" in part_df.columns and "et_gwsm_m" in part_df.columns:
    if "month" in part_df.columns:
        monthly = part_df.groupby(["water_year", "month"])[["et_gwsm_m", "et_irr_m"]].mean()
        monthly = monthly.reset_index()
        monthly["period"] = monthly["water_year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
        monthly = monthly.sort_values(["water_year", "month"])

        fig, ax = plt.subplots(figsize=(12, 5))
        x = range(len(monthly))
        ax.fill_between(x, monthly["et_gwsm_m"], alpha=0.6, color=PARTITION_COLORS["et_gwsm"], label="ET_gwsm")
        ax.fill_between(x, monthly["et_gwsm_m"] + monthly["et_irr_m"],
                        monthly["et_gwsm_m"], alpha=0.6, color=PARTITION_COLORS["et_irr"], label="ET_irr")
        ax.set_ylabel("mm/month")
        ax.set_title("Monthly ET Components (mean across fields)")
        ax.legend()

        # Label years on x-axis
        year_starts = monthly.groupby("water_year").apply(lambda g: g.index.min() - monthly.index.min())
        for yr, pos in year_starts.items():
            ax.axvline(pos, color="gray", linestyle="--", alpha=0.5, linewidth=0.8)
            ax.text(pos, ax.get_ylim()[1] * 0.95, str(yr), fontsize=8, ha="left")

        ax.set_xticks([])
        plt.tight_layout()
        plt.show()
    else:
        print("[INFO] 'month' column not found in partition data — skip monthly plot")
else:
    print("[SKIP] Partition data not yet computed — run partition_et first")

> **Operational Callout — QGIS Output**
>
> The partition parquet files are not directly loadable in QGIS, but the spatial
> summary can be joined back to fields and exported as FlatGeobuf:
>
> ```bash
> handily qgis update --config beaverhead_config.toml
> handily qgis qlr --config beaverhead_config.toml
> ```
>
> This updates `{config.qgis_project}` with all raster and vector outputs
> and generates a `.qlr` drag-and-drop layer definition file for the partition results.

> **Repo Capability — Notebook 5 (ET Partitioning)**
> Modules exercised: `handily.et.partition`, `handily.qgis`
> Key functions: `partition_et()`, `discover_outputs()`, `update_project()`, `generate_qlr()`
> CLI equivalents: `handily partition`, `handily qgis update`, `handily qgis qlr`

> **Artifacts Produced**
> - `{partition_out_parquet_dir}/{FID}.parquet` — monthly ET components (pe, et_gwsm_m, et_irr_m) per field per water year
> - `{qgis_project}` — updated QGIS project with all raster/vector layers
> - `*.qlr` — QGIS layer definition file for drag-and-drop use

## 7. Interpreting Results

### Output Columns

| Column | Description |
|--------|-------------|
| `water_year` | Water year (Oct-Sep) |
| `FID` | Field identifier |
| `donor_fid` | Donor (pattern) field ID |
| `fdisag` | Monthly disaggregation factor |
| `et_gwsm_m` | Monthly groundwater/soil moisture ET (mm) |
| `et_irr_m` | Monthly irrigation ET (mm) |

In [ ]:
# Visualize partitioned results
if part_dir and parquet_files:
    # Monthly time series
    part_df.index = pd.to_datetime(part_df.index)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    if 'et_gwsm_m' in part_df.columns and 'et_irr_m' in part_df.columns:
        # Stacked bar chart
        width = 20  # days
        ax.bar(part_df.index, part_df['et_gwsm_m'], width=width, 
               label='Groundwater/Soil Moisture', color='steelblue', alpha=0.7)
        ax.bar(part_df.index, part_df['et_irr_m'], width=width, 
               bottom=part_df['et_gwsm_m'], label='Irrigation', color='orange', alpha=0.7)
        
        ax.set_ylabel('ET (mm/month)')
        ax.set_xlabel('Date')
        ax.set_title('Monthly ET Partitioning')
        ax.legend()
    else:
        print("Expected columns not found in partitioned data")
    
    plt.tight_layout()

### Annual Summary

In [ ]:
if part_dir and parquet_files:
    if 'water_year' in part_df.columns and 'et_gwsm_m' in part_df.columns:
        annual = part_df.groupby('water_year').agg({
            'et_gwsm_m': 'sum',
            'et_irr_m': 'sum'
        })
        annual['total'] = annual['et_gwsm_m'] + annual['et_irr_m']
        annual['irr_fraction'] = annual['et_irr_m'] / annual['total']
        
        print("Annual Summary (mm/year):")
        print(annual.round(1))
        
        # Bar chart
        fig, ax = plt.subplots(figsize=(10, 6))
        
        x = np.arange(len(annual))
        width = 0.35
        
        ax.bar(x - width/2, annual['et_gwsm_m'], width, label='ET_gwsm', color='steelblue')
        ax.bar(x + width/2, annual['et_irr_m'], width, label='ET_irr', color='orange')
        
        ax.set_ylabel('ET (mm/year)')
        ax.set_xlabel('Water Year')
        ax.set_title('Annual ET Components')
        ax.set_xticks(x)
        ax.set_xticklabels(annual.index.astype(int))
        ax.legend()
        
        plt.tight_layout()

## 8. Validation considerations

Primary sensitivities: ETa uncertainty (model + Landsat sampling), donor representativeness, and local hydrogeologic heterogeneity. Useful checks include comparison to deliveries/pumping where available, known non-irrigated controls, and sensitivity to `rem_threshold`/pattern thresholds.

In [ ]:
# Compare multiple fields if available
if part_dir and len(parquet_files) > 1:
    # Load all partitioned files
    all_annual = []
    
    for f in parquet_files[:10]:  # Limit to first 10
        df = pd.read_parquet(os.path.join(part_dir, f))
        if 'water_year' in df.columns:
            annual = df.groupby('water_year').agg({
                'et_gwsm_m': 'sum',
                'et_irr_m': 'sum'
            }).mean()  # Average across years
            fid = f.replace('.parquet', '')
            all_annual.append({
                'FID': fid,
                'et_gwsm': annual['et_gwsm_m'],
                'et_irr': annual['et_irr_m']
            })
    
    if all_annual:
        compare_df = pd.DataFrame(all_annual)
        compare_df['total'] = compare_df['et_gwsm'] + compare_df['et_irr']
        compare_df['irr_pct'] = 100 * compare_df['et_irr'] / compare_df['total']
        
        print("\nComparison across fields (mm/year):")
        print(compare_df.round(1))
        
        # Histogram of irrigation fraction
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(compare_df['irr_pct'], bins=10, edgecolor='black')
        ax.set_xlabel('Irrigation % of Total ET')
        ax.set_ylabel('Number of Fields')
        ax.set_title('Distribution of Irrigation Fraction')
        plt.tight_layout()

## Key takeaways

- Donor-based `ET_gwsm` + residual `ET_irr` provides a pragmatic CU signal in shallow-GW settings.
- Results are best interpreted as relative differences/trends (field-to-field, year-to-year) unless independently calibrated.

For a scripted run, see `examples/beaverhead/beaverhead.py` and `examples/beaverhead/run_bvr.py`.